# XQuality Prefix Stop-After-Layer Evaluation — Exact CLI Evaluation with Normalized NeoOLAF Exports

This notebook evaluates already generated prefix stop-after-layer outputs.

It fixes the schema issue that produced zero scores for prefix outputs: the NeoOLAF CLI-compatible adapter expects triples in `kg_inferred.json` / `kg_local.json` with keys like `subject`, `predicate`, `object`, while the prefix generator may save `triples.json` with `subject_label`, `predicate`, `object_label`.

This notebook therefore:

1. Reads each completed prefix folder.
2. Converts its `triples.json` into a CLI-compatible NeoOLAF export folder.
3. Runs the same `evaluate_run(...)` path used by `python -m neoolaf.evaluation evaluate`.
4. Adds the native NeoOLAF final export as a reference row.
5. Saves summary CSVs and per-relation CSVs.

No OpenRouter calls are made here.

In [ ]:
from __future__ import annotations

import json
import re
import shutil
import warnings
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm

def find_repo_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "src" / "neoolaf").exists() and (p / "examples" / "XQualityMachine32").exists():
            return p
    raise RuntimeError(f"Could not find NeoOLAF repo root from {start}")

ROOT = find_repo_root()
print("ROOT:", ROOT)

import sys
src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from neoolaf.evaluation.runners.evaluate_run import evaluate_run
from neoolaf.evaluation.adapters.neoolaf_json_adapter import artifact_from_neoolaf_exports

In [ ]:
# =========================
# Configuration
# =========================

PROFILE_NAME = "xquality_relaxed_recall"

PREFIX_RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "prefix_stop_after_layer_llm_finalization_exact_eval"
EVAL_ROOT = ROOT / "examples" / "XQualityMachine32" / "runs" / "prefix_stop_after_layer_exact_cli_eval_normalized_exports"

GOLD_JSON = ROOT / "data" / "XQuality" / "Examples" / "XQuality_all_triplets_flat_en.json"

# Optional: set manually if auto-discovery picks the wrong native export folder.
# It should contain kg_local.json and/or kg_inferred.json.
NATIVE_FINAL_EXPORT_DIR = None

# If True, only evaluate folders with COMPLETED.json.
# If False, evaluate any folder with parsable triples.json.
REQUIRE_COMPLETED_MARKER = False

FORCE_REEVALUATE = True

print("PREFIX_RUN_DIR:", PREFIX_RUN_DIR, "| exists:", PREFIX_RUN_DIR.exists())
print("EVAL_ROOT:", EVAL_ROOT)
print("GOLD_JSON:", GOLD_JSON, "| exists:", GOLD_JSON.exists())
assert GOLD_JSON.exists(), f"Gold JSON not found: {GOLD_JSON}"
assert PREFIX_RUN_DIR.exists(), f"Prefix run dir not found: {PREFIX_RUN_DIR}"

In [ ]:
# =========================
# Helpers: parsing and normalization
# =========================

def safe_read_json(path: Path) -> Any:
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)

def clean_label(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, dict):
        for key in ["label", "text", "name", "value", "id"]:
            if key in x and x[key] not in (None, ""):
                return clean_label(x[key])
        return ""
    return re.sub(r"\s+", " ", str(x).strip())

def canonical_predicate(x: Any) -> str:
    s = clean_label(x).upper()
    s = re.sub(r"[^A-Z0-9]+", "_", s).strip("_")
    aliases = {
        "TRIGGER": "TRIGGERS",
        "TRIGGERS": "TRIGGERS",
        "CAUSE": "CAUSES",
        "CAUSES": "CAUSES",
        "REQUIRE": "REQUIRES",
        "REQUIRES": "REQUIRES",
        "HANDLED_BY": "HANDLED_BY",
        "HANDLES": "HANDLED_BY",
        "RESPONSIBLE": "HANDLED_BY",
        "REFERENCE": "REFERENCES",
        "REFERENCES": "REFERENCES",
        "REF": "REFERENCES",
    }
    return aliases.get(s, s)

def extract_triples_from_any_json(data: Any) -> list[dict[str, Any]]:
    records = []

    def visit(obj: Any):
        if isinstance(obj, dict):
            if isinstance(obj.get("triples"), list):
                for t in obj["triples"]:
                    visit(t)
                return

            subj = obj.get("subject_label", None)
            if subj is None:
                subj = obj.get("subject", obj.get("head", obj.get("s", None)))
            pred = obj.get("predicate", obj.get("predicate_label", obj.get("relation", obj.get("p", None))))
            tail = obj.get("object_label", None)
            if tail is None:
                tail = obj.get("object", obj.get("tail", obj.get("o", None)))

            subj_label = clean_label(subj)
            pred_label = canonical_predicate(pred)
            obj_label = clean_label(tail)

            if subj_label and pred_label and obj_label:
                records.append({
                    "subject_label": subj_label,
                    "predicate": pred_label,
                    "object_label": obj_label,
                    "confidence": obj.get("confidence"),
                    "evidence": obj.get("evidence") or obj.get("justification") or obj.get("support_text") or "",
                    "chunk_id": obj.get("chunk_id") or obj.get("chunkid") or "",
                    "raw": obj,
                })
                return

            for v in obj.values():
                visit(v)
        elif isinstance(obj, list):
            for item in obj:
                visit(item)

    visit(data)
    return records

def load_prefix_triples(folder: Path) -> list[dict[str, Any]]:
    candidates = [
        folder / "triples.json",
        folder / "llm_triples.json",
        folder / "kg.json",
        folder / "kg_local.json",
        folder / "kg_inferred.json",
    ]
    triples = []
    for path in candidates:
        if path.exists():
            try:
                triples.extend(extract_triples_from_any_json(safe_read_json(path)))
            except Exception as e:
                print(f"[WARN] Could not parse {path}: {e}")

    seen = set()
    out = []
    for t in triples:
        key = (t["subject_label"].lower(), t["predicate"].upper(), t["object_label"].lower())
        if key in seen:
            continue
        seen.add(key)
        out.append(t)
    return out

def to_neoolaf_export_triple(t: dict[str, Any]) -> dict[str, Any]:
    conf = t.get("confidence")
    if not isinstance(conf, (float, int)):
        conf = None

    return {
        "subject": {"label": t["subject_label"]},
        "predicate": t["predicate"],
        "object": {"label": t["object_label"]},
        "confidence": conf,
        "evidence": t.get("evidence") or "",
        "chunk_id": t.get("chunk_id") or "",
        "raw_source": t.get("raw", {}),
    }

def write_normalized_export_folder(source_folder: Path, triples: list[dict[str, Any]], normalized_folder: Path, run_id: str) -> Path:
    normalized_folder.mkdir(parents=True, exist_ok=True)

    export = {
        "document_id": "xquality",
        "run_id": run_id,
        "source_folder": str(source_folder),
        "triples": [to_neoolaf_export_triple(t) for t in triples],
    }

    for name in ["kg_inferred.json", "kg_local.json", "kg.json"]:
        (normalized_folder / name).write_text(
            json.dumps(export, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

    (normalized_folder / "triples_normalized_source_labels.json").write_text(
        json.dumps(triples, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    return normalized_folder

In [ ]:
# =========================
# Discover completed prefix output folders
# =========================

def parse_stop_index(name: str) -> int | None:
    m = re.search(r"prefix_stop_after_(\d+)_", name)
    return int(m.group(1)) if m else None

prefix_folders = []
for folder in sorted(PREFIX_RUN_DIR.glob("prefix_stop_after_*")):
    if not folder.is_dir():
        continue

    completed_marker = folder / "COMPLETED.json"
    has_marker = completed_marker.exists()
    if REQUIRE_COMPLETED_MARKER and not has_marker:
        print(f"[SKIP] {folder.name}: no COMPLETED.json")
        continue

    triples = load_prefix_triples(folder)
    if not triples:
        print(f"[SKIP] {folder.name}: no parsable triples")
        continue

    prefix_folders.append({
        "series": "prefix_stop_after_layer_generated_output",
        "layer_name": folder.name,
        "stop_index": parse_stop_index(folder.name),
        "input_folder": folder,
        "triple_count": len(triples),
        "has_completed_marker": has_marker,
    })

prefix_df = pd.DataFrame(prefix_folders)
if not prefix_df.empty:
    prefix_df = prefix_df.sort_values(["stop_index", "layer_name"], na_position="last").reset_index(drop=True)

display(prefix_df)
print("Completed/prefix folders to evaluate:", len(prefix_df))

In [ ]:
# =========================
# Native final export auto-discovery
# =========================

def looks_like_neoolaf_export_folder(path: Path) -> bool:
    return path.is_dir() and any((path / n).exists() for n in ["kg_inferred.json", "kg_local.json", "kg.json"])

def count_export_relations(path: Path) -> int:
    try:
        artifact = artifact_from_neoolaf_exports(path, dataset="xquality", profile=PROFILE_NAME, run_id=path.name)
        return sum(len(v) for v in artifact.relations_by_doc.values())
    except Exception:
        return 0

def find_native_export_candidates() -> list[Path]:
    roots = [
        ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32",
        ROOT / "examples" / "XQualityMachine32" / "runs",
    ]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        candidates.extend([p for p in root.rglob("exports") if looks_like_neoolaf_export_folder(p)])
        candidates.extend([p for p in root.rglob("*") if looks_like_neoolaf_export_folder(p)])

    uniq = []
    seen = set()
    for p in candidates:
        rp = p.resolve()
        if rp in seen:
            continue
        seen.add(rp)
        uniq.append(p)

    scored = []
    for p in uniq:
        rc = count_export_relations(p)
        try:
            mtime = max((p / n).stat().st_mtime for n in ["kg_inferred.json", "kg_local.json", "kg.json"] if (p / n).exists())
        except Exception:
            mtime = 0
        if rc > 0:
            scored.append((rc, mtime, p))

    scored.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return [p for _, _, p in scored]

if NATIVE_FINAL_EXPORT_DIR is not None:
    native_export_dir = Path(NATIVE_FINAL_EXPORT_DIR)
else:
    candidates = find_native_export_candidates()
    print("Native export candidates:")
    for i, p in enumerate(candidates[:10]):
        print(f"{i:02d} | relations={count_export_relations(p):4d} | {p}")
    native_export_dir = candidates[0] if candidates else None

print("\nSelected native_export_dir:", native_export_dir)
if native_export_dir is not None:
    print("Native relation count by adapter:", count_export_relations(native_export_dir))
else:
    print("[WARN] No native export dir found. Set NATIVE_FINAL_EXPORT_DIR manually if needed.")

In [ ]:
# =========================
# Exact CLI-compatible evaluation helpers
# =========================

def flatten_per_relation(per_relation: Any) -> list[dict[str, Any]]:
    out = []
    if isinstance(per_relation, dict):
        for pred, metrics in per_relation.items():
            row = {"predicate": pred}
            if isinstance(metrics, dict):
                row.update(metrics)
            out.append(row)
    elif isinstance(per_relation, list):
        for item in per_relation:
            if isinstance(item, dict):
                out.append(dict(item))
    return out

def metrics_row_from_summary(
    summary: dict[str, Any],
    series: str,
    layer_name: str,
    stop_index: int | None,
    profile: str,
    triple_count: int,
    source_folder: Path,
    normalized_input_folder: Path | None = None,
) -> dict[str, Any]:
    extraction = summary.get("extraction", {})
    ent = extraction.get("entity", {})
    rel = extraction.get("relation", {})
    counts = extraction.get("counts", {})

    return {
        "series": series,
        "layer_name": layer_name,
        "stop_index": stop_index,
        "profile": profile,
        "triple_count": triple_count,
        "source_folder": str(source_folder),
        "normalized_input_folder": str(normalized_input_folder) if normalized_input_folder else "",
        "pred_relations_seen_by_evaluator": counts.get("pred_relations"),
        "gold_relations_seen_by_evaluator": counts.get("gold_relations"),
        "pred_entities_seen_by_evaluator": counts.get("pred_entities"),
        "gold_entities_seen_by_evaluator": counts.get("gold_entities"),
        "entity_tp": ent.get("tp", 0),
        "entity_fp": ent.get("fp", 0),
        "entity_fn": ent.get("fn", 0),
        "entity_precision": ent.get("precision", 0),
        "entity_recall": ent.get("recall", 0),
        "entity_f1": ent.get("f1", 0),
        "relation_tp": rel.get("tp", 0),
        "relation_fp": rel.get("fp", 0),
        "relation_fn": rel.get("fn", 0),
        "relation_precision": rel.get("precision", 0),
        "relation_recall": rel.get("recall", 0),
        "relation_f1": rel.get("f1", 0),
    }

def run_eval_for_export_folder(input_folder: Path, eval_output_dir: Path, run_id: str, modality: str) -> dict[str, Any]:
    if FORCE_REEVALUATE and eval_output_dir.exists():
        shutil.rmtree(eval_output_dir)
    eval_output_dir.mkdir(parents=True, exist_ok=True)

    return evaluate_run(
        dataset="xquality",
        method="neoolaf",
        profile_name=PROFILE_NAME,
        gold_path=GOLD_JSON,
        output_dir=eval_output_dir,
        input_path=input_folder,
        run_id=run_id,
        modality=modality,
        ontology_path=None,
    )

In [ ]:
# =========================
# Prepare normalized inputs and evaluate
# =========================

EVAL_ROOT.mkdir(parents=True, exist_ok=True)
NORMALIZED_INPUTS_ROOT = EVAL_ROOT / "normalized_prefix_exports"
EVAL_OUTPUTS_ROOT = EVAL_ROOT / "eval_outputs"

rows = []
per_relation_rows = []

for rec in tqdm(prefix_folders, desc="Evaluating normalized prefix folders"):
    folder = Path(rec["input_folder"])
    triples = load_prefix_triples(folder)

    normalized_folder = NORMALIZED_INPUTS_ROOT / rec["layer_name"]
    write_normalized_export_folder(
        source_folder=folder,
        triples=triples,
        normalized_folder=normalized_folder,
        run_id=rec["layer_name"],
    )

    adapter_rel_count = count_export_relations(normalized_folder)
    print(f"{rec['layer_name']}: source triples={len(triples)} | adapter relations={adapter_rel_count}")
    if len(triples) > 0 and adapter_rel_count == 0:
        raise RuntimeError(
            f"Normalized export schema still not readable by adapter for {rec['layer_name']}: {normalized_folder}"
        )

    summary = run_eval_for_export_folder(
        input_folder=normalized_folder,
        eval_output_dir=EVAL_OUTPUTS_ROOT / rec["layer_name"],
        run_id=rec["layer_name"],
        modality="prefix_stop_after_layer_generated_output",
    )

    row = metrics_row_from_summary(
        summary=summary,
        series=rec["series"],
        layer_name=rec["layer_name"],
        stop_index=rec["stop_index"],
        profile=PROFILE_NAME,
        triple_count=len(triples),
        source_folder=folder,
        normalized_input_folder=normalized_folder,
    )
    rows.append(row)

    extraction = summary.get("extraction", {})
    for pr in flatten_per_relation(extraction.get("per_relation", {})):
        pr.update({
            "series": rec["series"],
            "layer_name": rec["layer_name"],
            "stop_index": rec["stop_index"],
            "profile": PROFILE_NAME,
        })
        per_relation_rows.append(pr)

if native_export_dir is not None:
    native_summary = run_eval_for_export_folder(
        input_folder=Path(native_export_dir),
        eval_output_dir=EVAL_OUTPUTS_ROOT / "native_neoolaf_final_export",
        run_id="native_neoolaf_final_export",
        modality="native_neoolaf_final_export",
    )
    native_triples = count_export_relations(Path(native_export_dir))
    rows.append(metrics_row_from_summary(
        summary=native_summary,
        series="native_neoolaf_final_export",
        layer_name="native_neoolaf_final_export",
        stop_index=12,
        profile=PROFILE_NAME,
        triple_count=native_triples,
        source_folder=Path(native_export_dir),
        normalized_input_folder=None,
    ))
    extraction = native_summary.get("extraction", {})
    for pr in flatten_per_relation(extraction.get("per_relation", {})):
        pr.update({
            "series": "native_neoolaf_final_export",
            "layer_name": "native_neoolaf_final_export",
            "stop_index": 12,
            "profile": PROFILE_NAME,
        })
        per_relation_rows.append(pr)

summary_df = pd.DataFrame(rows)
per_relation_df = pd.DataFrame(per_relation_rows)

summary_path = EVAL_ROOT / "summary_exact_cli_normalized_exports.csv"
per_relation_path = EVAL_ROOT / "per_relation_exact_cli_normalized_exports.csv"
summary_df.to_csv(summary_path, index=False)
per_relation_df.to_csv(per_relation_path, index=False)

print("Saved:", summary_path)
print("Saved:", per_relation_path)

display(summary_df.sort_values(["series", "stop_index"], na_position="last"))

In [ ]:
# =========================
# Sanity checks
# =========================

display_cols = [
    "series", "layer_name", "stop_index", "profile", "triple_count",
    "pred_relations_seen_by_evaluator", "gold_relations_seen_by_evaluator",
    "relation_precision", "relation_recall", "relation_f1",
    "entity_f1", "relation_tp", "relation_fp", "relation_fn",
]
display(summary_df[display_cols].sort_values("relation_f1", ascending=False))

if not summary_df.empty:
    gold_counts = set(summary_df["gold_relations_seen_by_evaluator"].dropna().astype(int).tolist())
    print("Gold relation counts seen:", gold_counts)
    assert all(x > 0 for x in gold_counts), "Gold relation count is zero somewhere: invalid evaluation."

bad_prefix = summary_df[
    (summary_df["series"] == "prefix_stop_after_layer_generated_output")
    & (summary_df["triple_count"] > 0)
    & (summary_df["pred_relations_seen_by_evaluator"].fillna(0).astype(int) == 0)
]
if len(bad_prefix):
    display(bad_prefix)
    raise RuntimeError("Some prefix folders had triples but evaluator saw zero predicted relations.")

native_rows = summary_df[summary_df["series"] == "native_neoolaf_final_export"]
if len(native_rows):
    native = native_rows.iloc[0]
    print("\nNative final export sanity:")
    print(native[["relation_precision", "relation_recall", "relation_f1", "relation_tp", "relation_fp", "relation_fn"]])
    if native["relation_precision"] < 0.90 or native["relation_f1"] < 0.55:
        warnings.warn(
            "Native final export is not close to expected final-export score. "
            "Check NATIVE_FINAL_EXPORT_DIR, GOLD_JSON, and PROFILE_NAME."
        )

In [ ]:
# =========================
# Plot
# =========================

import matplotlib.pyplot as plt

plot_df = summary_df.copy()
plot_df = plot_df.sort_values(["series", "stop_index"])

plt.figure(figsize=(12, 6))
for series, group in plot_df.groupby("series"):
    if series == "prefix_stop_after_layer_generated_output":
        g = group.dropna(subset=["stop_index"]).sort_values("stop_index")
        plt.plot(g["stop_index"], g["relation_f1"], marker="o", label="Prefix stop-after-layer output")
    elif series == "native_neoolaf_final_export":
        y = group["relation_f1"].iloc[0]
        plt.axhline(y=y, linestyle="--", label=f"Native NeoOLAF final export F1={y:.3f}")

plt.xlabel("Stop-after layer index")
plt.ylabel("Relation F1")
plt.title("Prefix stop-after-layer finalization vs native NeoOLAF final export")
plt.legend()
plt.grid(True, alpha=0.3)
plot_path = EVAL_ROOT / "relation_f1_prefix_stop_after_layer.png"
plt.savefig(plot_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved plot:", plot_path)